# Assignment 2: The Dirty Ledger — work notebook
Audit of `teaching_prices_messy.csv` (weekly illustrative prices, 4 items x 2 cities x 12 weeks).
**Data label:** the file is *ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS*; nothing here describes real prices.

Structure: (1) load and diagnose, (2) run the AI's cleaning code on a copy, (3) my cleaning on a separate copy, (4) before/after table and checks.
The original `df_raw` is never modified; every cleaning step works on `.copy()`.

## 1. Load and diagnose (before changing anything)

In [2]:
import pandas as pd
pd.set_option('display.width', 200)

df_raw = pd.read_csv('teaching_prices_messy.csv')
print('shape:', df_raw.shape)
print()
df_raw.info()

shape: (97, 6)

<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        97 non-null     str  
 1   item        97 non-null     str  
 2   city        97 non-null     str  
 3   price       96 non-null     str  
 4   unit        97 non-null     str  
 5   data_label  97 non-null     str  
dtypes: str(6)
memory usage: 4.7 KB


In [3]:
print('Missing values per column:')
print(df_raw.isna().sum())
print()
print('Rows with a missing price:')
print(df_raw[df_raw['price'].isna()][['date','item','city','price']])

Missing values per column:
date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64

Rows with a missing price:
         date     item    city price
5  2026-01-01  Pulse C  City 2   NaN


In [4]:
print('item values:');  print(df_raw['item'].value_counts()); print()
print('city values:');  print(df_raw['city'].value_counts()); print()
print('unit values:');  print(df_raw['unit'].value_counts()); print()
print('data_label values:'); print(df_raw['data_label'].value_counts())

item values:
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int64

city values:
city
City 1    49
City 2    48
Name: count, dtype: int64

unit values:
unit
PKR per kg (illustrative)    97
Name: count, dtype: int64

data_label values:
data_label
ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS    97
Name: count, dtype: int64


In [5]:
# price is text (object) because at least one cell is not a plain number. Which ones?
numeric = pd.to_numeric(df_raw['price'], errors='coerce')
print('Not parseable as a plain number (and not blank):')
print(df_raw[numeric.isna() & df_raw['price'].notna()][['date','item','city','price']])
print()
print('describe() of the parseable prices (note the max):')
print(numeric.describe())
print()
print('Values above 500:')
print(df_raw[numeric > 500][['date','item','city','price']])

Not parseable as a plain number (and not blank):
          date      item    city  price
17  2026-01-15  Staple A  City 2  1,250

describe() of the parseable prices (note the max):
count      95.000000
mean      305.621053
std      1008.261424
min        99.500000
25%       135.500000
50%       200.000000
75%       301.000000
max      9999.000000
Name: price, dtype: float64

Values above 500:
          date         item    city price
30  2026-01-22  Sweetener D  City 1  9999


In [6]:
print('Exact duplicate rows:')
print(df_raw[df_raw.duplicated(keep=False)][['date','item','city','price']])
print()
print('Expected: 12 dates x 4 items x 2 cities = 96 rows; each combination once.')
counts = df_raw.groupby(['item','city']).size()
print(counts)

Exact duplicate rows:
          date      item    city price
8   2026-01-08  Staple A  City 1  99.5
96  2026-01-08  Staple A  City 1  99.5

Expected: 12 dates x 4 items x 2 cities = 96 rows; each combination once.
item         city  
Oil B        City 1    12
             City 2    12
Pulse C      City 1    12
             City 2    12
Staple A     City 1    12
             City 2    12
Sweetener D  City 1    12
             City 2    12
oil b        City 1     1
dtype: int64


In [7]:
# The lower-case 'oil b' row: what is it really?
print(df_raw[(df_raw['date'] == '2026-02-05')][['date','item','city','price']])
print()
print('Staple A, City 1 around that date:')
print(df_raw[(df_raw['item'] == 'Staple A') & (df_raw['city'] == 'City 1') & (df_raw['date'].between('2026-01-29','2026-02-12'))][['date','item','city','price']])

          date         item    city  price
40  2026-02-05        oil b  City 1  108.5
41  2026-02-05     Staple A  City 2  114.5
42  2026-02-05        Oil B  City 1  318.0
43  2026-02-05        Oil B  City 2  324.0
44  2026-02-05      Pulse C  City 1  202.5
45  2026-02-05      Pulse C  City 2  208.5
46  2026-02-05  Sweetener D  City 1  161.5
47  2026-02-05  Sweetener D  City 2  167.5

Staple A, City 1 around that date:
          date      item    city  price
32  2026-01-29  Staple A  City 1  105.0
48  2026-02-12  Staple A  City 1  107.0


In [8]:
# Evidence for the 'oil b' row: in every clean date/item, City 2 is City 1 plus 6.
tmp = df_raw.copy()
tmp['p'] = pd.to_numeric(tmp['price'].astype(str).str.replace(',', ''), errors='coerce')
tmp.loc[tmp['item'] == 'oil b', 'item'] = 'Staple A'          # test the hypothesis
tmp = tmp.drop_duplicates()
pv = tmp.pivot_table(index=['date','item'], columns='city', values='p')
gap = (pv['City 2'] - pv['City 1']).round(2)
print('City 2 minus City 1 across all date/item pairs (hypothesis: relabelled row is Staple A):')
print(gap.value_counts(dropna=False))

City 2 minus City 1 across all date/item pairs (hypothesis: relabelled row is Staple A):
 6.0       45
 NaN        1
 1147.0     1
-9835.5     1
Name: count, dtype: int64


In [9]:
dates = pd.to_datetime(df_raw['date'])
print('distinct dates:', dates.nunique(), '| weekdays:', dates.dt.day_name().unique(), '| range:', dates.min().date(), 'to', dates.max().date())

distinct dates: 12 | weekdays: <StringArray>
['Thursday']
Length: 1, dtype: str | range: 2026-01-01 to 2026-03-19


## 2. The AI's cleaning code, run as-is on a copy
This is the code an AI tool suggested for my plain-words description (columns, types and the problems). Each numbered line is judged in the log file (`3_log_reflection.md`).

In [11]:
ai = df_raw.copy()

ai.columns = ai.columns.str.strip().str.lower()                                    # L1
ai['item'] = ai['item'].str.strip().str.title()                                    # L2
ai['price'] = ai['price'].astype(str).str.replace(',', '').astype(float)           # L3
ai = ai.drop_duplicates()                                                          # L4
ai = ai.dropna(subset=['price'])                                                   # L5
ai = ai[ai['price'] < 1000]                                                        # L6
ai['date'] = pd.to_datetime(ai['date'])                                            # L7
ai = ai.drop(columns=['unit', 'data_label'])                                       # L8

print('AI result shape:', ai.shape)
print(ai.groupby(['item','city']).size())

AI result shape: (93, 4)
item         city  
Oil B        City 1    13
             City 2    12
Pulse C      City 1    12
             City 2    11
Staple A     City 1    11
             City 2    11
Sweetener D  City 1    11
             City 2    12
dtype: int64


In [12]:
# Step-by-step row counts for the AI code (to give each line its own evidence)
step = df_raw.copy()
log = [('start', len(step))]
step['item'] = step['item'].str.strip().str.title();                      log.append(('L2 title-case item', len(step)))
step['price'] = step['price'].astype(str).str.replace(',', '').astype(float); log.append(('L3 strip commas', len(step)))
n = len(step); step = step.drop_duplicates();                              log.append(('L4 drop_duplicates (removed %d)' % (n-len(step)), len(step)))
n = len(step); step = step.dropna(subset=['price']);                       log.append(('L5 dropna (removed %d)' % (n-len(step)), len(step)))
n = len(step); step = step[step['price'] < 1000];                          log.append(('L6 price<1000 (removed %d)' % (n-len(step)), len(step)))
for name, rows in log: print(f'{name:40s} rows = {rows}')
print()
print('Side effect of L2: the relabelled row now collides with a real Oil B row.')
print(step[(step['item'] == 'Oil B') & (step['city'] == 'City 1') & (step['date'] == '2026-02-05')][['date','item','city','price']])
print()
print('Rows the AI removed that were NOT exact duplicates:')
gone = df_raw.copy()
gone['price_num'] = pd.to_numeric(gone['price'].astype(str).str.replace(',', ''), errors='coerce')
print(gone[(gone['price_num'].isna()) | (gone['price_num'] >= 1000)][['date','item','city','price']])

start                                    rows = 97
L2 title-case item                       rows = 97
L3 strip commas                          rows = 97
L4 drop_duplicates (removed 1)           rows = 96
L5 dropna (removed 1)                    rows = 95
L6 price<1000 (removed 2)                rows = 93

Side effect of L2: the relabelled row now collides with a real Oil B row.
          date   item    city  price
40  2026-02-05  Oil B  City 1  108.5
42  2026-02-05  Oil B  City 1  318.0

Rows the AI removed that were NOT exact duplicates:
          date         item    city  price
5   2026-01-01      Pulse C  City 2    NaN
17  2026-01-15     Staple A  City 2  1,250
30  2026-01-22  Sweetener D  City 1   9999


## 3. My cleaning (on a separate copy)
Rules: fix only what the evidence supports; flag what it does not; never invent a number; keep an audit trail.

In [14]:
clean = df_raw.copy()
clean = clean.rename(columns={'price': 'price_raw'})
clean['price_raw'] = clean['price_raw'].astype('string')          # keep the original text for the audit trail
clean['flag'] = ''

# --- P1: lower-case 'oil b' is a mislabelled Staple A / City 1 row (see evidence above) ---
m = (clean['item'] == 'oil b') & (clean['date'] == '2026-02-05') & (clean['city'] == 'City 1') & (clean['price_raw'] == '108.5')
assert m.sum() == 1
clean.loc[m, 'item'] = 'Staple A'
clean.loc[m, 'flag'] = 'relabelled: item was "oil b", evidence = price level + missing Staple A slot + City gap of 6'

# --- P2: exact duplicate row (2026-01-08 Staple A City 1) -> keep first ---
before = len(clean)
clean = clean.drop_duplicates(subset=['date','item','city','price_raw','unit','data_label'], keep='first')
assert before - len(clean) == 1

# --- parse price: remove thousands separators, keep NaN where unparseable ---
clean['price'] = pd.to_numeric(clean['price_raw'].str.replace(',', '', regex=False), errors='coerce')

# --- P3: blank price -> keep the row, flag it, do NOT impute ---
miss = clean['price'].isna()
clean.loc[miss, 'flag'] = 'missing price in source; not imputed'

# --- P4 / P5: implausible values (1,250 and the 9999 code) -> keep the row, blank the price, flag it ---
big = clean['price'] > 500
assert big.sum() == 2
clean.loc[big & (clean['price'] == 9999), 'flag'] = '9999 looks like a missing-value code; price set to NaN'
clean.loc[big & (clean['price'] == 1250), 'flag'] = '"1,250" is ~11x neighbours (City 1 = 103); true value unknown; price set to NaN'
clean.loc[big, 'price'] = float('nan')

# --- dates to real dates; keep unit and data_label (they carry the ILLUSTRATIVE warning) ---
clean['date'] = pd.to_datetime(clean['date'])
clean = clean[['date','item','city','price','price_raw','unit','data_label','flag']].sort_values(['date','item','city']).reset_index(drop=True)

print(clean.shape)
print()
print('Every changed or flagged row:')
print(clean[clean['flag'] != ''][['date','item','city','price','price_raw','flag']].to_string())

(96, 8)

Every changed or flagged row:
         date         item    city  price price_raw                                                                                          flag
3  2026-01-01      Pulse C  City 2   <NA>      <NA>                                                          missing price in source; not imputed
21 2026-01-15     Staple A  City 2   <NA>     1,250               "1,250" is ~11x neighbours (City 1 = 103); true value unknown; price set to NaN
30 2026-01-22  Sweetener D  City 1   <NA>      9999                                        9999 looks like a missing-value code; price set to NaN
44 2026-02-05     Staple A  City 1  108.5     108.5  relabelled: item was "oil b", evidence = price level + missing Staple A slot + City gap of 6


## 4. Before and after

In [16]:
def summary(d, price_col, label):
    return pd.Series({
        'rows': len(d),
        'missing price': int(d[price_col].isna().sum()),
        'min': pd.to_numeric(d[price_col], errors='coerce').min(),
        'max': pd.to_numeric(d[price_col], errors='coerce').max(),
        'mean': round(pd.to_numeric(d[price_col], errors='coerce').mean(), 2),
        'distinct items': d['item'].nunique(),
    }, name=label)

before_num = df_raw.copy(); before_num['price'] = pd.to_numeric(before_num['price'].astype(str).str.replace(',', ''), errors='coerce')
table = pd.concat([
    summary(before_num, 'price', 'raw (commas stripped to read)'),
    summary(ai, 'price', 'AI code as-is'),
    summary(clean, 'price', 'my cleaning'),
], axis=1)
print(table)

                raw (commas stripped to read)  AI code as-is  my cleaning
rows                                    97.00          93.00        96.00
missing price                            1.00           0.00         3.00
min                                     99.50          99.50        99.50
max                                   9999.00         350.00       350.00
mean                                   315.46         203.61       203.61
distinct items                           5.00           4.00         4.00


In [17]:
print('describe() before (commas stripped):'); print(before_num['price'].describe().round(2)); print()
print('describe() after (mine):');             print(clean['price'].describe().round(2))

describe() before (commas stripped):
count      96.00
mean      315.46
std      1007.56
min        99.50
25%       142.25
50%       200.00
75%       304.00
max      9999.00
Name: price, dtype: float64

describe() after (mine):
count      93.0
mean     203.61
std       79.92
min        99.5
25%       149.0
50%       200.0
75%       298.0
max       350.0
Name: price, dtype: Float64


In [18]:
# Does every item/city appear the expected 12 times?
def cnt(d): return d.groupby(['item','city']).size().unstack()
print('RAW:'); print(cnt(df_raw)); print()
print('AI (as-is):'); print(cnt(ai)); print()
print('MINE:'); print(cnt(clean))

RAW:
city         City 1  City 2
item                       
Oil B          12.0    12.0
Pulse C        12.0    12.0
Staple A       12.0    12.0
Sweetener D    12.0    12.0
oil b           1.0     NaN

AI (as-is):
city         City 1  City 2
item                       
Oil B            13      12
Pulse C          12      11
Staple A         11      11
Sweetener D      11      12

MINE:
city         City 1  City 2
item                       
Oil B            12      12
Pulse C          12      12
Staple A         12      12
Sweetener D      12      12


In [19]:
# Integrity checks on my result
assert len(clean) == 96
assert (cnt(clean) == 12).all().all()
assert not clean.duplicated(subset=['date','item','city']).any()
assert set(clean['item']) == {'Staple A','Oil B','Pulse C','Sweetener D'}
assert clean['price'].isna().sum() == 3
assert clean['data_label'].str.contains('ILLUSTRATIVE').all()

# No valid observation lost or changed: compare every unflagged row to the raw file.
raw_chk = df_raw.copy()
raw_chk['p'] = pd.to_numeric(raw_chk['price'], errors='coerce')
raw_chk = raw_chk.drop_duplicates()
ok = clean[clean['flag'] == '']
raw_chk['date'] = pd.to_datetime(raw_chk['date'])
merged = ok.merge(raw_chk[['date','item','city','p']], on=['date','item','city'], how='left')
print('unflagged rows compared with raw:', len(merged))
print('rows where cleaned price != raw price:', int((merged['price'].round(2) != merged['p'].round(2)).sum()))
print('All checks passed.')

unflagged rows compared with raw: 92
rows where cleaned price != raw price: 0
All checks passed.


In [20]:
# Save the cleaned file (original stays untouched)
clean.to_csv('teaching_prices_clean.csv', index=False)
print('saved teaching_prices_clean.csv', clean.shape)

saved teaching_prices_clean.csv (96, 8)
